# Demo legal reasoning theo idea từ paper [MedAgents](https://ar5iv.labs.arxiv.org/html/2311.10537#:~:text=Image%3A%20Refer%20to%20caption%20Figure,a%20final%20decision%2C%20reflecting%20unanimous)

## 1. Import libraries and API KEY

In [1]:
from llm import *
from prompt import *
from generate import *
from types import SimpleNamespace

## 2. Generate answers

### Sample question/context/answer

In [2]:
question = 'Does the clause waive damages?'
context = 'This Agreement may be terminated by either party for any reason by giving the other party thirty (30) days written notice of such termination.,mismatch.'

In [11]:
question = f'{context}' \
            f'{question}' \

options = 'Yes/ No'
gold_answer = 'No'


### Direct Answering
- Ý tưởng: gọi thẳng prompt để LLM trả về ngay kết quả (Yes hoặc No)

In [10]:
args = SimpleNamespace(method="base_direct", max_attempt_vote=3)
result = fully_decode(question, options, gold_answer, args)
print("=== Full Decode Result ===")
print("Question Domains:", result["question_domains"])
print("Option Domains:", result["option_domains"])
print("Question Analyses:", result["question_analyses"])
print("Option Analyses:", result["option_analyses"])
print("Synthesized Report:", result["syn_report"])
print("Vote History:", result["vote_history"])
print("Revision History:", result["revision_history"])
print("Synthesized Reports History:", result["syn_repo_history"])
print("Output:", result["raw_output"])

Finish generating response in 0.5 seconds
=== Full Decode Result ===
Question Domains: []
Option Domains: []
Question Analyses: {}
Option Analyses: {}
Synthesized Report: 
Vote History: []
Revision History: []
Synthesized Reports History: []
Output: Option: No


### Chain-of-thought
- ý tưởng: yêu cầu LLM think step by step để tăng độ chính xác và suy luận

In [15]:
args = SimpleNamespace(method="base_cot", max_attempt_vote=3)
result = fully_decode(question, options, gold_answer, args)
print("=== Full Decode Result ===")
print("Predicted Answer:", result["pred_answer"])
print("Question Domains:", result["question_domains"])
print("Option Domains:", result["option_domains"])
print("Question Analyses:", result["question_analyses"])
print("Option Analyses:", result["option_analyses"])
print("Synthesized Report:", result["syn_report"])
print("Vote History:", result["vote_history"])
print("Revision History:", result["revision_history"])
print("Synthesized Reports History:", result["syn_repo_history"])
print("Output:", result["raw_output"])

Finish generating response in 3.76 seconds
=== Full Decode Result ===
Predicted Answer: 
Question Domains: []
Option Domains: []
Question Analyses: {}
Option Analyses: {}
Synthesized Report: 
Vote History: []
Revision History: []
Synthesized Reports History: []
Output: To determine if the clause waives damages, let's analyze it step by step:

1. **Understanding the Clause**: The clause states that either party can terminate the agreement for any reason by providing thirty (30) days written notice to the other party. This implies a mutual right to terminate without specifying any conditions or penalties for termination, other than the notice period.

2. **Termination for Any Reason**: The phrase "for any reason" suggests that the termination is not contingent upon a breach of contract or any other specific condition. This broad language does not inherently imply a waiver of damages but rather focuses on the freedom to terminate.

3. **Notice Period**: The requirement for a thirty (30) d

### Anal-only
- Ý tưởng: gồm 3 steps:
    - Phân loại question vào 5 domains nhỏ trong luật
    - Phân tích question theo từng sub field
    - Phân tích đáp án theo 2 subfields phù hợp nhất đc chọn sau khi phân tích -> gọi prompt để chọn đáp án

In [14]:
args = SimpleNamespace(method="anal_only", max_attempt_vote=3)
result = fully_decode(question, options, gold_answer, args)
print("=== Full Decode Result ===")
print("Predicted Answer:", result["pred_answer"])
print("Question Domains:", result["question_domains"])
print("Option Domains:", result["option_domains"])
print("Question Analyses:", result["question_analyses"])
print("Option Analyses:", result["option_analyses"])
print("Synthesized Report:", result["syn_report"])
print("Vote History:", result["vote_history"])
print("Revision History:", result["revision_history"])
print("Synthesized Reports History:", result["syn_repo_history"])
print("Output:", result["raw_output"])

Finish generating response in 1.0 seconds
Finish generating response in 1.08 seconds
Finish generating response in 3.73 seconds
Finish generating response in 3.85 seconds
Finish generating response in 0.55 seconds
=== Full Decode Result ===
Predicted Answer: 
Question Domains: ['The scenario describes a contractual clause that allows either party to terminate the agreement for any reason, provided they give the other party thirty (30']
Option Domains: ['This field is crucial because the scenario involves an agreement between two parties, specifically focusing on the termination clause. Contract']
Question Analyses: {'The scenario describes a contractual clause that allows either party to terminate the agreement for any reason, provided they give the other party thirty (30': 'The contractual clause in question allows either party to terminate the agreement for any reason, provided they give the other party thirty (30) days written notice of such termination. This type of clause is commo

### Syn_only:
- Ý tưởng: Sau khi thực hiện pipeline **Anal-only** thì theme bước tổng hợp tất cả các phân tích đó thành report, gửi thêm report này vào prompt để chọn đáp án

In [7]:
args = SimpleNamespace(method="syn_only", max_attempt_vote=3)
result = fully_decode(question, options, gold_answer, args)
print("=== Full Decode Result ===")
print("Question Domains:", result["question_domains"])
print("Option Domains:", result["option_domains"])
print("Question Analyses:", result["question_analyses"])
print("Option Analyses:", result["option_analyses"])
print("Synthesized Report:", result["syn_report"])
print("Vote History:", result["vote_history"])
print("Revision History:", result["revision_history"])
print("Synthesized Reports History:", result["syn_repo_history"])
print("Output:", result["raw_output"])

Finish generating response in 1.01 seconds
Finish generating response in 1.0 seconds
Finish generating response in 3.12 seconds
Finish generating response in 4.02 seconds
Finish generating response in 2.09 seconds
Finish generating response in 0.51 seconds
=== Full Decode Result ===
Question Domains: ["The given clause pertains to the termination of an agreement, allowing either party to terminate the agreement by providing thirty (30) days' written"]
Option Domains: ['This field is crucial because the scenario involves an agreement between two parties, which is a fundamental aspect of contract law']
Question Analyses: {"The given clause pertains to the termination of an agreement, allowing either party to terminate the agreement by providing thirty (30) days' written": 'The given clause pertains to the termination of an agreement, allowing either party to terminate the agreement by providing thirty (30) days\' written notice. This is a common provision in contracts, enabling parties t

### Tổng hợp cuối cùng:
- Ý tưởng: Thực hiện **Anal_only** và **Syn_only**, sau đó thêm bước voting và chỉnh sửa lại kết quả, sau đó gọi llm để đưa ra report và kết quả cuối cùng

In [9]:
args = SimpleNamespace(method="syn_verif", max_attempt_vote=3)
result = fully_decode(question, options, gold_answer, args)
print("=== Full Decode Result ===")
print("Question Domains:", result["question_domains"])
print("Option Domains:", result["option_domains"])
print("Question Analyses:", result["question_analyses"])
print("Option Analyses:", result["option_analyses"])
print("Synthesized Report:", result["syn_report"])
print("Vote History:", result["vote_history"])
print("Revision History:", result["revision_history"])
print("Synthesized Reports History:", result["syn_repo_history"])
print("Output:", result["raw_output"])

Finish generating response in 1.02 seconds
Finish generating response in 1.05 seconds
Finish generating response in 3.73 seconds
Finish generating response in 3.87 seconds
Finish generating response in 2.08 seconds
Finish generating response in -0.12 seconds
Finish generating response in 3.22 seconds
Finish generating response in 0.49 seconds
Finish generating response in 2.58 seconds
Finish generating response in 4.37 seconds
Finish generating response in 0.49 seconds
Finish generating response in 3.41 seconds
Finish generating response in 0.49 seconds
Finish generating response in 3.32 seconds
Finish generating response in 8.24 seconds
Finish generating response in 0.53 seconds
Finish generating response in 3.64 seconds
Finish generating response in 0.5 seconds
Finish generating response in 3.53 seconds
Finish generating response in 9.66 seconds
Finish generating response in 0.53 seconds
=== Full Decode Result ===
Question Domains: ["The given clause pertains to the termination of an